# 13.05 · Voice architecture: latency budgets, streaming, turn-taking, and telephony

**Module 13 · Multimodal & Voice** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2 h |
| 📋 Prerequisites | Lesson 13.04 (the batch voice sandwich). Read the module README first. |
| 🔑 Keys | none for the core (offline); optional `GOOGLE_API_KEY` for the 💸 Gemini Live probe (free tier) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** a latency budget for one voice turn and compare it with what you measured in 13.04.
2. **Explain** partial vs final transcripts and why an agent acts only on the final one.
3. **Tune** endpointing (when has the caller finished?) with an energy VAD and see the trade-off of `silence_ms`.
4. **Simulate** barge-in: the caller talks over the agent, playback stops and the agent knows what was heard.
5. **Compare** a cascaded pipeline (STT → LLM → TTS) with speech-to-speech models, and measure one (optional).
6. **Survey** what production adds: telephony (SIP/PSTN), WebRTC, and frameworks such as LiveKit Agents and Pipecat.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Build a voice agent with LangChain](https://docs.langchain.com/oss/python/langchain/voice-agent) | Streaming sandwich, turn handling, latency claim |
| [Gemini Live API](https://ai.google.dev/gemini-api/docs/live) | Speech-to-speech over a WebSocket, voice activity detection, interruptions |
| [LiveKit Agents](https://docs.livekit.io/agents/) | WebRTC rooms, turn detection, telephony (SIP) |
| [Pipecat](https://docs.pipecat.ai/getting-started/introduction) | Frame pipelines for voice bots, interruption handling |

> **Why this matters at work.** The batch sandwich of 13.04 answers in 2–4 seconds and can't be interrupted. Callers notice both. This lesson is the design review you'd do before putting a voice agent on a real phone number: where the milliseconds go, when to start answering, what happens when the caller talks over you, and which parts to buy instead of build.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "google-genai>=2.25.0,<3" "numpy>=2.3.0,<3"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("google-genai", "numpy",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/13-multimodal-and-voice/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY',)
LANGSMITH_PROJECT = 'm13-voice-architecture'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_GOOGLE_KEY = is_set("GOOGLE_API_KEY") or is_set("GEMINI_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m13-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("vision    :", settings.model_vision, "(Groq, preview)")
print("multimodal:", settings.model_multimodal)
print("stt / tts :", settings.model_stt, "/", settings.tts_provider)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_GOOGLE_KEY:
    print("⚠️ No GOOGLE_API_KEY: 💸 Gemini cells will print ⏭️ Skipped (free key: https://aistudio.google.com/apikey).")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models (qwen3.8 vision, Orpheus) warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

from mastery.voice import (Endpointer, EnergyVad, LatencyBudget, PartialTranscripts, Playback, frames,
                           frame_energy, speech_clip)

RUN_STRETCH = False   # set True to run the 💸 Gemini Live probe in Part 6 (free tier, 1 request)

---
## Part 1 · The latency budget

A real-time turn has stages the batch version didn't: waiting to be sure the caller has finished (**endpointing**), and network hops for each streamed stage. Targets for a turn that feels natural (under about 1.2 s from end of speech to first audio):

In [ ]:
# 🧪 Offline: targets per stage vs a batch turn like the one you timed in 13.04 (replace with your numbers).
budget = LatencyBudget({"endpoint": 500, "stt final": 200, "agent first sentence": 350, "tts first audio": 150})
batch_13_04 = {"endpoint": 0, "stt final": 700, "agent first sentence": 1300, "tts first audio": 900}
table(budget.report(batch_13_04), ["stage", "target ms", "measured ms", "ok"])

**You should now see** a table with `TOTAL` 1200 target against 2900 measured, `ok` False for the three provider stages.

### What just happened?

The batch turn has **no endpointing cost** (the file just ends), but every other stage waits for the previous one to finish. Streaming attacks each row: STT streams partials so the final arrives ~200 ms after speech ends; the agent's first sentence comes from a small fast model; TTS starts playing the first chunk. Endpointing then becomes the biggest single item, and it's a product decision (Part 3).

---
## Part 2 · Partial and final transcripts

Streaming STT sends a growing **partial** hypothesis while the caller talks, then a **final** one when the turn ends. Partials can change their last words:

In [ ]:
# 🧪 Offline: what a streaming STT shows while the caller says one sentence.
said = "please unlock the account for customer C-1001"
for t, text, final in PartialTranscripts(said, revise={"account": "a count", "customer": "custom"}).stream():
    print(f"{t:4.1f} s  {'FINAL  ' if final else 'partial'}  {text}")

**You should now see** partial lines that grow, at least one with a wrong last word (`custom` instead of `customer`), and a `FINAL` line equal to the sentence.

### What just happened?

Partials are for the **screen** (live captions) and for **early work** that is safe to throw away (pre-fetching the customer record, warming the model). Tool calls, refunds and unlocks wait for the **final** transcript: acting on `a count` would be a bug you can't reproduce from the logs.

---
## Part 3 · Endpointing: has the caller finished?

A **VAD** (voice activity detection) labels each 20 ms frame as speech or not. An **endpointer** declares the turn over after `silence_ms` of silence. Here is a synthetic caller who pauses 400 ms to think in the middle of the question:

In [ ]:
# 🧪 Offline: one question with a thinking pause, over light line noise.
caller = speech_clip([("silence", 0.5), ("speech", 1.4), ("silence", 0.4), ("speech", 1.0), ("silence", 1.2)], noise_db=-50)
levels = [round(frame_energy(f)) for f in frames(caller, 100)]
print("loudness per 100 ms (dBFS):", levels)
for silence_ms in (200, 400, 600, 900):
    events = Endpointer(silence_ms=silence_ms).run(caller)
    turns = [e.at_s for e in events if e.kind == "end_of_turn"]
    delay = round((turns[-1] - 3.3) * 1000) if turns else None
    print(f"silence_ms={silence_ms:4}: {len(turns)} turn(s) ended at {turns}  → answer can start {delay} ms after the caller stops")

**You should now see** `silence_ms=200` (and usually 400) splitting the question into **2 turns** (the agent would interrupt at the pause), while 600 and 900 give **1 turn**, ending about 600 and 900 ms after the caller stops.

### What just happened?

That's the endpointing trade-off: short = snappy but cuts thinkers off; long = polite but every millisecond is added to the answer. Production systems mix signals: a trained VAD (Silero, WebRTC), a **semantic** turn detector (does the sentence sound finished?), and a shorter timeout after a question mark. LiveKit and Pipecat ship turn-detector models for exactly this.

---
## Part 4 · Barge-in: the caller talks over the agent

The agent is reading four sentences; 1.5 s in, the caller says "no, the other invoice". The player must **stop**, and the agent must know what the caller actually heard:

In [ ]:
# 🧪 Offline: play an answer in chunks; the caller interrupts at 1.5 s.
answer = ["Your latest invoice is INV-2026-0912.", "It totals 1,240 dollars.", "It is due on November 5.",
          "You can pay it from the billing page."]
vad = Endpointer(silence_ms=600)
interrupt = speech_clip([("silence", 1.5), ("speech", 0.6)], noise_db=-55)
starts = [e.at_s for e in vad.run(interrupt) if e.kind == "speech_start"]
playback = Playback(answer, seconds_per_chunk=1.0)
heard = playback.play(interrupt_at_s=starts[0] if starts else None)
print("caller started talking at:", starts[0] if starts else None, "s; playback stopped at:", playback.cancelled_at_s, "s")
print("heard  :", heard)
print("unheard:", playback.unheard)
history = [{"role": "assistant", "content": " ".join(heard) + " [interrupted]"}]
print("history:", history)

**You should now see** the caller detected at about 1.5 s, playback stopped at 2.0 s, two sentences heard and two unheard, and a history entry ending in `[interrupted]`.

### What just happened?

Three things a barge-in handler must do: **stop audio** at the next chunk boundary (smaller chunks = faster stop), **cancel** the TTS and agent work still in flight, and **rewrite history** to what was heard. If history keeps the full answer, the agent later says "as I mentioned, it's due November 5" when the caller never heard it. Echo matters too: on a speakerphone the agent's own voice can trigger the VAD, so real systems use echo cancellation (WebRTC does it in the browser).

---
## Part 5 · Cascaded vs speech-to-speech

| | Cascaded (STT → LLM → TTS, this module) | Speech-to-speech (Gemini Live, OpenAI Realtime) |
|---|---|---|
| Latency | Sum of stages; 0.8–1.5 s streamed | One model; often under 1 s |
| Tools, RAG, your LangChain agent | Any agent, any model, full control | Tool calling supported, but the model is fixed and the loop lives in the session |
| Transcripts and audit | Text at every step, easy to log and test | Ask for transcriptions; the reasoning is audio-native |
| Voice and prosody | Pick any TTS voice | Expressive, reacts to tone of voice |
| Interruptions | You build them (Part 4) | Built in (server-side VAD) |
| Cost and lock-in | Swap any stage | One provider for the whole call |

A common split: speech-to-speech for open conversation, cascaded when every answer must come from your tools with a text audit trail (support, banking, health).

---
## Part 6 · (Optional) Measure a speech-to-speech model: Gemini Live

One question over the Live API, measuring time to first audio. Set `RUN_STRETCH = True` in the setup cell above to run it.

> 💸 Calls Gemini on the **free tier** (no charge, daily request caps shown in AI Studio). Free-tier content may be used by Google to improve its products: the course sends synthetic samples only. Without `GOOGLE_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 (RUN_STRETCH) One Gemini Live turn: text in, audio out, with a transcript of what it said.
import asyncio
import time

LIVE_MODEL = os.environ.get("MASTERY_MODEL_LIVE", "gemini-3.8-live")

async def live_probe(question: str) -> dict:
    from google import genai
    client = genai.Client()
    config = {"response_modalities": ["AUDIO"], "output_audio_transcription": {},
              "system_instruction": "You are a phone support agent for Orbita Cloud. Answer in two short sentences."}
    t0, first, audio_bytes, words = time.perf_counter(), None, 0, []
    async with client.aio.live.connect(model=LIVE_MODEL, config=config) as session:
        await session.send_realtime_input(text=question)
        async for message in session.receive():
            content = message.server_content
            if not content:
                continue
            for part in (content.model_turn.parts if content.model_turn else []):
                if part.inline_data and part.inline_data.data:
                    audio_bytes += len(part.inline_data.data)
                    first = first or time.perf_counter() - t0
            if content.output_transcription and content.output_transcription.text:
                words.append(content.output_transcription.text)
            if content.turn_complete:
                break
    return {"first audio s": round(first or 0, 2), "total s": round(time.perf_counter() - t0, 2),
            "audio KB": audio_bytes // 1024, "said": "".join(words).strip()}

if RUN_STRETCH and HAVE_GOOGLE_KEY:
    try:
        probe = await live_probe("Is the station sync service down right now?")
        print(f"Live first audio: {probe['first audio s']} s, total {probe['total s']} s, {probe['audio KB']} KB")
        print("said:", probe["said"])
    except Exception as error:  # noqa: BLE001 - preview models move; show the reason instead of failing Run All
        print(f"❌ Gemini Live probe failed on {LIVE_MODEL}: {type(error).__name__}: {str(error)[:300]}")
        print("   Check the model name on the Live API page (override with MASTERY_MODEL_LIVE).")
elif RUN_STRETCH:
    print("⏭️ Skipped: GOOGLE_API_KEY is not set (see the setup cell).")
else:
    print("⏭️ Skipped (optional): set RUN_STRETCH = True in the setup cell to run the Gemini Live probe.")

**You should now see** `⏭️ Skipped (optional)` by default. With `RUN_STRETCH = True` and a key: `Live first audio: ...`, typically under 1 s, and the transcript of the spoken answer. Compare it with the first audio you measured in 13.04 (sequential and sentences).

### What just happened?

One WebSocket session did listening, thinking and speaking. The first audio chunk arrives before the model has finished the sentence. You didn't run your desk's tools, though: to use them you'd declare them in the session config and answer tool calls yourself, inside the Live loop.

---
## Part 7 · Telephony and frameworks: what to buy

| Need | Typical choice |
|---|---|
| A phone number (PSTN) and SIP trunking | Twilio, Telnyx, Vonage; or the SIP integration of LiveKit |
| Browser and app audio with echo cancellation | WebRTC (LiveKit, Daily) rather than raw WebSockets |
| Turn detection, interruptions, pipeline plumbing | [LiveKit Agents](https://docs.livekit.io/agents/) or [Pipecat](https://docs.pipecat.ai/), both open source, both able to call a LangChain/LangGraph agent |
| Streaming STT / TTS | Deepgram, AssemblyAI, Cartesia, ElevenLabs, Gemini; Groq offers batch Whisper and Orpheus |

What stays **yours** whatever you buy: the agent and its tools (Project 09), identity rules (13.04 Part 6), the spoken-reply style (`for_voice`), budgets and fallbacks (13.03), and the evaluation set of recorded calls (Project 13).

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Agent answers in the middle of the caller's sentence | `silence_ms` too short | 500–800 ms, or a semantic turn detector |
| Long pause before every answer | `silence_ms` too long, or STT final waits for the whole clip | Shorter timeout after a finished question; streaming STT |
| Agent "hears itself" and stops talking | Its own audio triggers the VAD | Echo cancellation (WebRTC), or ignore VAD while playing on half-duplex lines |
| Agent repeats things the caller never heard | History keeps the full interrupted answer | Rewrite history to the played chunks + `[interrupted]` |
| Tool fired on a wrong word | Acted on a partial transcript | Tools only on the final transcript |
| Everything is still silent in a noisy room | Fixed `min_db` threshold | Adaptive floor (`EnergyVad` calibrates on the first 200 ms) or a trained VAD |
| Live probe errors with "model not found" | Preview model names change | Check the Live API page; set `MASTERY_MODEL_LIVE` |

## ⚠️ Production notes

> ⚠️ **Production note: measure the turn the caller feels.** (Parts 1, 3)
> Log end-of-speech → first-audio for every turn and alert on its p95, not only per-provider latency. Endpointing time is part of it.

> ⚠️ **Production note: speech-to-speech and preview models.** (Part 6)
> Live models change names and behaviour often. Keep a cascaded fallback and record the model version per call. Free-tier traffic may be used to improve Google's products: send only synthetic or consented audio.

## 🏋️ Exercises

1. **(Easy)** Make the thinking pause in Part 3 800 ms. Which `silence_ms` values still give one turn?
2. **(Medium)** In Part 4, make each chunk half a sentence (`seconds_per_chunk=0.5`, eight chunks). How much earlier does playback stop?
3. **(Medium)** Fill `batch_13_04` with your real numbers from 13.04 (sequential and sentences) and print both budget tables.
4. **(Stretch)** Write `smart_endpoint(text, silence_ms)` that ends the turn after 300 ms if the partial transcript ends with `?`, else after 800 ms. Test it on two `PartialTranscripts`.

<details>
<summary><b>Show solutions</b></summary>

**1.** `speech_clip([("silence", 0.5), ("speech", 1.4), ("silence", 0.8), ("speech", 1.0), ("silence", 1.2)])`: only `silence_ms=900` keeps one turn; 600 now splits it.

**2.** With 0.5 s chunks playback stops at 1.5 s instead of 2.0 s: smaller chunks, faster barge-in, more TTS requests.

**3.** `table(budget.report({"endpoint": 0, "stt final": stt_ms, "agent first sentence": agent_ms, "tts first audio": tts_ms}), ...)` with the `timings` dict from a `VoiceTurn`.

**4.**
```python
def smart_endpoint(partial: str) -> int:
    return 300 if partial.rstrip().endswith("?") else 800

for said in ("is the sync service down?", "my invoice is wrong and"):
    last_partial = list(PartialTranscripts(said).stream())[-1][1]
    print(f"{said!r} → end the turn after {smart_endpoint(last_partial)} ms of silence")
```
</details>

## 🔎 Docs drill (5 min)

In the Gemini Live API, what does the server do by default when the user starts speaking while the model is answering?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open the [Live API](https://ai.google.dev/gemini-api/docs/live) guide and find the voice activity detection section.
2. Automatic VAD is on by default: when it detects user speech, the current generation is **interrupted** and discarded, and the server reports the interruption.
3. You can turn automatic VAD off and send activity start/end signals yourself.

</details>

## ✅ Checkpoint

- [ ] I filled a latency budget and know which stage dominates my turn.
- [ ] I can explain why tools wait for the final transcript.
- [ ] I saw `silence_ms` split or keep a turn, and picked a value.
- [ ] I simulated barge-in and rewrote history to what was heard.
- [ ] I can argue cascaded vs speech-to-speech for a support line.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check the [Live API](https://ai.google.dev/gemini-api/docs/live) page and log in [`docs/research-log.md`](../../docs/research-log.md): the current Live model names, whether they have a free tier, and (if you ran Part 6) your measured first audio next to 13.04's.

## Summary

- A real-time turn = endpointing + streaming STT + agent first sentence + TTS first audio; target about 1.2 s.
- Partials are for captions and safe early work; tools act on the final transcript.
- `silence_ms` trades interrupting thinkers against slow answers; production adds semantic turn detection.
- Barge-in: stop playback, cancel work, and keep in history only what was heard.
- Speech-to-speech is fastest; cascaded keeps your tools and a text audit trail. Buy telephony and WebRTC; keep the agent yours.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 13 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)